# Notebook 6: User Network Statistics (Table E.2)

Computes basic network statistics for the user-user LCC graphs.

**Inputs** from `../data/processed/`:
- `user_graph_{key}.pkl` — user-user LCC graphs

**Output** (displayed inline):
- Table E.2 — user network basic statistics

> **Runtime warning**: This notebook is intentionally kept separate because Louvain community detection on medium-event user networks (up to 192k nodes, 12M edges) is computationally expensive. **Allow 2–4 hours** for a full run, or run interactively one dataset at a time. Average shortest path and diameter are skipped for medium networks (marked `*` in the thesis).

In [1]:
import pickle
import random
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd

DATA_DIR = Path('../data/processed')

KEYS = [
    'small_2013-2014', 'small_2014-2015', 'small_2015-2016', 'small_2016-2017',
    'medium_2013-2014', 'medium_2014-2015', 'medium_2015-2016', 'medium_2016-2017',
]

def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

user_graphs = {k: load_pkl(DATA_DIR / f'user_graph_{k}.pkl') for k in KEYS}
print("Loaded all user-user graphs.")

Loaded all user-user graphs.


In [ ]:
def approx_path_stats(G, n_sample=1000, seed=42):
    """Estimate average shortest path and diameter by BFS from n_sample nodes."""
    rng    = random.Random(seed)
    nodes  = list(G.nodes())
    sample = rng.sample(nodes, min(n_sample, len(nodes)))
    dists  = []
    for src in sample:
        d = nx.single_source_shortest_path_length(G, src)
        dists.extend(v for v in d.values() if v > 0)
    return round(np.mean(dists), 2), max(dists)


table_e2_rows = []

for size in ['Small', 'Medium']:
    for period in ['2013-2014', '2014-2015', '2015-2016', '2016-2017']:
        key = f'{size.lower()}_{period}'
        G   = user_graphs[key]
        print(f"\n{key} ({G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges)")

        print("  clustering...")
        clustering = round(nx.average_clustering(G, weight='weight'), 2)

        print("  density...")
        density = round(nx.density(G), 4)

        print("  assortativity...")
        assortativity = round(nx.degree_assortativity_coefficient(G), 2)

        print("  Louvain communities...")
        communities = nx.community.louvain_communities(G, seed=42)
        modularity  = round(nx.community.modularity(G, communities), 4)

        if size == 'Small':
            print("  path stats...")
            avg_path, diameter = approx_path_stats(G)
        else:
            avg_path, diameter = '*', '*'

        table_e2_rows.append({
            'Event Size': size, 'Time Period': period,
            'Users': G.number_of_nodes(), 'Edges': G.number_of_edges(),
            'Clustering Coeff': clustering,
            'Density': density,
            'Degree Assortativity': assortativity,
            'Modularity': modularity,
            'Avg Shortest Path (approx)': avg_path,
            'Diameter (approx)': diameter,
        })
        print(f"  done: CC={clustering}, mod={modularity}")

table_e2 = pd.DataFrame(table_e2_rows)
print("\nTable E.2 — User Networks LCC Basic Statistics:")
print(table_e2.to_string(index=False))


small_2013-2014 (62,534 nodes, 867,261 edges)
  clustering...
  density...
  assortativity...
  Louvain communities...
  path stats...
  done: CC=0.02, mod=0.7861

small_2014-2015 (72,066 nodes, 1,038,418 edges)
  clustering...
  density...
  assortativity...
  Louvain communities...
  path stats...
  done: CC=0.02, mod=0.7667

small_2015-2016 (99,209 nodes, 1,484,858 edges)
  clustering...
  density...
  assortativity...
  Louvain communities...
  path stats...
  done: CC=0.01, mod=0.7236

small_2016-2017 (113,432 nodes, 1,676,789 edges)
  clustering...
  density...
  assortativity...
  Louvain communities...
  path stats...
  done: CC=0.01, mod=0.6976

medium_2013-2014 (126,726 nodes, 7,463,539 edges)
  clustering...
  density...
  assortativity...
  Louvain communities...
  done: CC=0.02, mod=0.6283

medium_2014-2015 (150,725 nodes, 9,522,664 edges)
  clustering...
  density...
  assortativity...
  Louvain communities...
  done: CC=0.02, mod=0.5913

medium_2015-2016 (189,586 nodes,